# Bagging, Boosting, KNN and Stacking

Assignment answers  |  Source: ml8.pdf

All questions are answered in slightly expanded, student-friendly language. Code blocks are runnable examples; dataset-dependent outputs should be generated locally.

## Question 1. What is the fundamental idea behind ensemble techniques? How does bagging differ from boosting in terms of approach and objective?

Answer: Ensemble methods combine multiple learners to improve accuracy, robustness or stability. Bagging trains models independently on bootstrap samples and aggregates predictions, mainly reducing variance; Random Forest is an example. Boosting trains learners sequentially to focus on errors or residuals, often reducing bias and building a strong additive model.

## Question 2. Explain how the Random Forest Classifier reduces overfitting compared to a single decision tree. Mention the role of two key hyperparameters in this process.

Answer: Random Forest averages predictions from many decorrelated trees trained on bootstrap samples and random subsets of features. This usually reduces variance compared with one deep tree. n_estimators controls how many trees are averaged, while max_depth limits each tree’s complexity; min_samples_leaf can further reduce overfitting.

## Question 3. What is Stacking in ensemble learning? How does it differ from traditional bagging/boosting methods? Provide a simple example use case.

Answer: Stacking combines predictions from several base models using a meta-model trained on out-of-fold predictions. Unlike bagging, its models need not be the same type; unlike boosting, they are not necessarily trained sequentially. For house-price prediction, a meta-regressor might combine Random Forest, Ridge and gradient-boosting predictions.

## Question 4. What is the OOB Score in Random Forest, and why is it useful? How does it help in model evaluation without a separate validation set?

Answer: Out-of-bag (OOB) observations are training samples omitted from an individual tree’s bootstrap draw. Their predictions can be aggregated across trees where they were excluded to estimate generalization performance without reserving an extra validation split. OOB is convenient but does not replace a final independent test set.

## Question 5. Compare AdaBoost and Gradient Boosting in terms of: ●​ How they handle errors from weak learners ●​ Weight adjustment mechanism ●​ Typical use cases

Answer: AdaBoost increases the influence of previously misclassified training observations when fitting successive weak learners. Gradient Boosting instead fits each new learner to the negative gradient of a differentiable loss, often interpreted as residual correction for squared error. AdaBoost is a classic approach for simple classifiers; gradient boosting is flexible for regression and classification with complex tabular patterns.

## Question 6. Why does CatBoost perform well on categorical features without requiring extensive preprocessing? Briefly explain its handling of categorical variables.

Answer: CatBoost handles categorical features through ordered target statistics and carefully designed boosting procedures. It can use categorical columns without extensive manual one-hot encoding and aims to reduce target leakage with ordered calculations. This is especially useful for tabular datasets containing many category values.

## Question 7. KNN Classifier Assignment: Wine Dataset Analysis with Optimization Task: 1.​ Load the Wine dataset (sklearn.datasets.load_wine()). 2.​ Split data into 70% train and 30% test. 3.​ Train a KNN classifier (default K=5) without scaling and evaluate using: a.​ Accuracy b.​ Precision, Recall, F1-Score (print classification report) 4.​ Apply StandardScaler, retrain KNN, and compare metrics. 5.​ Use GridSearchCV to find the best K (test K=1 to 20) and distance metric (Euclidean, Manhattan). 6.​ Train the optimized KNN and compare results with the unscaled/scaled versions.

Answer: Compare unscaled KNN, standardized KNN and a tuned standardized pipeline. Euclidean and Manhattan distances are searched without leaking scaling information into validation folds.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
X,y=load_wine(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
models={'Unscaled':KNeighborsClassifier(n_neighbors=5),
        'Scaled':make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5))}
for name,model in models.items():
    model.fit(Xtr,ytr); print(name,model.score(Xte,yte))
    print(classification_report(yte,model.predict(Xte)))
pipe=make_pipeline(StandardScaler(),KNeighborsClassifier())
search=GridSearchCV(pipe,{'kneighborsclassifier__n_neighbors':list(range(1,21)),
    'kneighborsclassifier__metric':['euclidean','manhattan']},cv=5)
search.fit(Xtr,ytr)
print('Best:',search.best_params_,'Test:',search.score(Xte,yte))

## Question 8. PCA + KNN with Variance Analysis and Visualization Task: 1.​ Load the Breast Cancer dataset (sklearn.datasets.load_breast_cancer()). 2.​ Apply PCA and plot the scree plot (explained variance ratio). 3.​ Retain 95% variance and transform the dataset. 4.​ Train KNN on the original data and PCA-transformed data, then compare accuracy. 5.​ Visualize the first two principal components using a scatter plot (color by class).

Answer: PCA reduces dimensionality by retaining directions of high variance. Standardize features first, fit PCA on training data only, and compare KNN accuracy on standardized original features and components retaining 95% variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
X,y=load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
sc=StandardScaler().fit(Xtr); A=sc.transform(Xtr); B=sc.transform(Xte)
full=PCA().fit(A)
plt.plot(np.arange(1,len(full.explained_variance_ratio_)+1),full.explained_variance_ratio_)
plt.title('PCA scree plot'); plt.xlabel('Component'); plt.show()
pca=PCA(n_components=.95).fit(A)
for name,train,test in [('Original',A,B),('PCA',pca.transform(A),pca.transform(B))]:
    model=KNeighborsClassifier().fit(train,ytr)
    print(name,'accuracy:',model.score(test,yte))
points=PCA(n_components=2).fit_transform(A)
plt.scatter(points[:,0],points[:,1],c=ytr,s=10,cmap='viridis')
plt.xlabel('PC1');plt.ylabel('PC2');plt.show()

## Question 9. KNN Regressor with Distance Metrics and K-Value Analysis Task: 1.​ Generate a synthetic regression dataset (sklearn.datasets.make_regression(n_samples=500, n_features=10)). 2.​ Train a KNN regressor with: a.​ Euclidean distance (K=5) b.​ Manhattan distance (K=5) c.​ Compare Mean Squared Error (MSE) for both. 3.​ Test K=1, 5, 10, 20, 50 and plot K vs. MSE to analyze bias-variance tradeoff.

Answer: Compare Euclidean and Manhattan distances for K=5, then plot test MSE for several K values. A very small K can overfit, while a very large K may smooth away important patterns.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
X,y=make_regression(n_samples=500,n_features=10,random_state=42)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
for metric in ['euclidean','manhattan']:
    m=make_pipeline(StandardScaler(),KNeighborsRegressor(n_neighbors=5,metric=metric))
    m.fit(Xtr,ytr); print(metric,'MSE:',mean_squared_error(yte,m.predict(Xte)))
ks=[1,5,10,20,50]; errors=[]
for k in ks:
    m=make_pipeline(StandardScaler(),KNeighborsRegressor(n_neighbors=k))
    m.fit(Xtr,ytr);errors.append(mean_squared_error(yte,m.predict(Xte)))
plt.plot(ks,errors,marker='o');plt.xlabel('K');plt.ylabel('Test MSE');plt.show()

## Question 10. KNN with KD-Tree/Ball Tree, Imputation, and Real-World Data Task: 1.​ Load the Pima Indians Diabetes dataset (contains missing values). 2.​ Use KNN Imputation (sklearn.impute.KNNImputer) to fill missing values. 3.​ Train KNN using: a.​ Brute-force method b.​ KD-Tree c.​ Ball Tree 4.​ Compare their training time and accuracy. 5.​ Plot the decision boundary for the best-performing method (use 2 most important features). Dataset: Pima Indians Diabetes

Answer: Download the Pima Indians Diabetes CSV separately: it is not included with the PDF. The code treats impossible zero readings as missing for selected medical measurements, uses training-only KNN imputation, and compares neighbor-search algorithms. The 2-feature boundary is a separate visualization model, not the full-feature model.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
# CSV columns should include Outcome and standard Pima features.
df=pd.read_csv('diabetes.csv')
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    df[col]=df[col].replace(0,np.nan)
X=df.drop(columns='Outcome');y=df['Outcome']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
results={}
for algorithm in ['brute','kd_tree','ball_tree']:
    model=make_pipeline(KNNImputer(n_neighbors=5),StandardScaler(),
                        KNeighborsClassifier(n_neighbors=5,algorithm=algorithm))
    start=time.perf_counter();model.fit(Xtr,ytr)
    elapsed=time.perf_counter()-start
    acc=accuracy_score(yte,model.predict(Xte))
    results[algorithm]=(elapsed,acc)
print(results)
# For a 2D decision boundary, retrain on Glucose and BMI only.